# HoTHP — Grid Search de Learning Rate (Amazon)

**Objetivo:** Encontrar o melhor learning rate para o HoTHP no dataset Amazon,
que foi onde o modelo se destacou mais no benchmark geral.

**Estratégia:**
- Modelo: HoTHP apenas
- Dataset: amazon (`train_max_len=18`, `eval_1x=18`, `eval_5x=90`)
- LRs testados: `[1e-3, 5e-4, 1e-4, 5e-5]` (1e-3 = baseline do benchmark anterior)
- 3 seeds por LR → 12 runs de treino, 24 runs de eval (1x + 5x)
- Métrica principal: **NLL 5x** (cenário onde HoTHP mostrou maior vantagem)

**Run on Colab:** Runtime → Change runtime type → T4 GPU

In [ ]:
import os, sys

# ── Clona o repositório ufc-easytpp ───────────────────────────────────
if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git

sys.path.insert(0, 'ufc-easytpp')

# ── Instala dependências ──────────────────────────────────────────────
!pip install omegaconf datasets pyyaml matplotlib pandas seaborn tqdm -q

# ── Fix: __init__.py mínimo ───────────────────────────────────────────
_init_path = 'ufc-easytpp/easy_tpp/model/__init__.py'
with open(_init_path, 'w') as f:
    f.write(
        "from easy_tpp.model.torch_model.torch_basemodel import TorchBaseModel\n"
        "from easy_tpp.model.torch_model.torch_hothp  import HoTHP  as TorchHoTHP\n"
    )

# ── Fix: força import dos modelos no runner ───────────────────────────
_runner_path = 'ufc-easytpp/easy_tpp/runner/tpp_runner.py'
with open(_runner_path, 'r') as f:
    _content = f.read()
if 'import easy_tpp.model' not in _content:
    _content = _content.replace(
        'from collections import OrderedDict\n',
        'from collections import OrderedDict\nimport easy_tpp.model  # noqa: F401\n'
    )
    with open(_runner_path, 'w') as f:
        f.write(_content)

import torch
GPU = 0 if torch.cuda.is_available() else -1
print(f'OK — GPU={GPU}, torch={torch.__version__}')

In [ ]:
# ── Informações de hardware ───────────────────────────────────────────
import platform, psutil, datetime, json

hw = {}
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    hw['gpu_name']     = props.name
    hw['gpu_vram_gb']  = round(props.total_memory / 1024**3, 2)
    hw['cuda_version'] = torch.version.cuda
else:
    hw['gpu_name']     = 'CPU only'
    hw['gpu_vram_gb']  = 0
    hw['cuda_version'] = 'N/A'

hw['cpu']       = platform.processor() or platform.machine()
hw['ram_gb']    = round(psutil.virtual_memory().total / 1024**3, 2)
hw['torch']     = torch.__version__
hw['timestamp'] = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print('╔══════════════════════════════════════════════╗')
print('║            HARDWARE DO AMBIENTE              ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  GPU        : {hw["gpu_name"]:<30} ║')
print(f'║  VRAM       : {str(hw["gpu_vram_gb"]) + " GB":<30} ║')
print(f'║  CUDA       : {hw["cuda_version"]:<30} ║')
print(f'║  CPU        : {hw["cpu"][:30]:<30} ║')
print(f'║  RAM        : {str(hw["ram_gb"]) + " GB":<30} ║')
print(f'║  PyTorch    : {hw["torch"]:<30} ║')
print(f'║  Início     : {hw["timestamp"]:<30} ║')
print('╚══════════════════════════════════════════════╝')

with open('hardware_info_lr_search.json', 'w') as f:
    json.dump(hw, f, indent=2)
print('\nSalvo: hardware_info_lr_search.json')

In [ ]:
import gc, pickle, tempfile, time, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from easy_tpp.config_factory import Config
from easy_tpp.runner import Runner

sns.set_theme(style='whitegrid')

# ── Parâmetros fixos ──────────────────────────────────────────────────
MODEL_ID      = 'HoTHP'
DATASET_NAME  = 'amazon'
SEEDS         = [2019, 2020, 2021]
MAX_EPOCH     = 100
BATCH_SIZE    = 256
HIDDEN_SIZE   = 64
NUM_HEADS     = 2
NUM_LAYERS    = 2
DROPOUT       = 0.1
TIME_EMB_SIZE = 16

# ── Grid de learning rates ────────────────────────────────────────────
# 1e-3 = baseline do benchmark anterior (incluído para referência direta)
LEARNING_RATES = [1e-3, 5e-4, 1e-4, 5e-5]

# ── Config do dataset amazon ──────────────────────────────────────────
DATASET_CFG = {
    'num_event_types': 16,
    'pad_token_id':    16,
    'train_max_len':   18,
    'eval_1x':         18,
    'eval_5x':         90,
}

CHECKPOINT_FILE = 'hothp_lr_search_progress.pkl'

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

print(f'Grid search: {MODEL_ID} × {DATASET_NAME}')
print(f'LRs    : {LEARNING_RATES}')
print(f'Seeds  : {SEEDS}')
print(f'Runs   : {len(LEARNING_RATES) * len(SEEDS)} treinos, '
      f'{len(LEARNING_RATES) * len(SEEDS) * 2} evals (1x + 5x)')

In [ ]:
# ── Funções auxiliares ────────────────────────────────────────────────

def lr_tag(lr):
    """Converte LR para string legível: 0.001 → '1e-3'."""
    return f'{lr:.0e}'


def load_progress():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'rb') as f:
            data = pickle.load(f)
        print(f'Progresso carregado: {len(data["results"])} evals, '
              f'{len(data["trained"])} modelos treinados')
        return data
    return {'results': [], 'trained': set(), 'train_times': {}}


def save_progress(progress):
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(progress, f)


def get_model_dir(lr, seed):
    """Retorna o checkpoint mais recente para (lr, seed)."""
    base = f'./checkpoints/{DATASET_NAME}/{MODEL_ID}/lr{lr_tag(lr)}/seed{seed}'
    if not os.path.isdir(base):
        return f'{base}/models/saved_model'
    candidates = []
    for entry in os.scandir(base):
        if entry.is_dir():
            candidate = os.path.join(entry.path, 'models', 'saved_model')
            if os.path.exists(candidate):
                candidates.append((entry.stat().st_mtime, candidate))
    if candidates:
        return sorted(candidates)[-1][1]
    return f'{base}/models/saved_model'


def make_yaml(lr, seed, max_len, stage='train', pretrained_model_dir=None):
    exp_id = f'{MODEL_ID}_{stage}'

    model_cfg = {
        'hidden_size':   HIDDEN_SIZE,
        'num_heads':     NUM_HEADS,
        'num_layers':    NUM_LAYERS,
        'dropout':       DROPOUT,
        'time_emb_size': TIME_EMB_SIZE,
        'use_ln':        False,
        'thinning': {
            'num_sample': 1, 'num_exp': 500, 'look_ahead_time': 10,
            'patience_counter': 5, 'over_sample_rate': 5,
            'num_samples_boundary': 5, 'dtime_max': 10,
            'num_seq': 10, 'num_step_gen': 1,
        },
    }
    if stage == 'train':
        model_cfg['loss_integral_num_sample_per_step'] = 20
        model_cfg['mc_num_sample_per_step'] = 20
    if pretrained_model_dir:
        model_cfg['pretrained_model_dir'] = pretrained_model_dir

    trainer_cfg = {
        'batch_size': BATCH_SIZE,
        'max_epoch':  MAX_EPOCH if stage == 'train' else 1,
        'seed':       seed,
        'gpu':        GPU,
        'metrics':    ['acc', 'rmse'],
    }
    if stage == 'train':
        trainer_cfg.update({
            'valid_freq': 1, 'use_tfb': False,
            'optimizer': 'adam', 'learning_rate': lr,
            'shuffle': False,
        })

    config = {
        'pipeline_config_id': 'runner_config',
        'data': {
            DATASET_NAME: {
                'data_format': 'json',
                'train_dir':   f'easytpp/{DATASET_NAME}',
                'valid_dir':   f'easytpp/{DATASET_NAME}',
                'test_dir':    f'easytpp/{DATASET_NAME}',
                'data_specs': {
                    'num_event_types':     DATASET_CFG['num_event_types'],
                    'pad_token_id':        DATASET_CFG['pad_token_id'],
                    'padding_side':        'right',
                    'truncation_side':     'right',
                    'truncation_strategy': 'longest_first',
                    'max_len':             max_len,
                },
            },
        },
        exp_id: {
            'base_config': {
                'stage':      stage,
                'backend':    'torch',
                'dataset_id': DATASET_NAME,
                'runner_id':  'std_tpp',
                'model_id':   MODEL_ID,
                'base_dir':   f'./checkpoints/{DATASET_NAME}/{MODEL_ID}/lr{lr_tag(lr)}/seed{seed}/',
            },
            'trainer_config': trainer_cfg,
            'model_config':   model_cfg,
        },
    }
    return config, exp_id


def write_yaml_and_load(config_dict, experiment_id):
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
        yaml.dump(config_dict, f, default_flow_style=False)
        tmp_path = f.name
    try:
        cfg = Config.build_from_yaml_file(tmp_path, experiment_id=experiment_id)
    finally:
        os.unlink(tmp_path)
    return cfg


print('Funções auxiliares definidas.')

In [ ]:
# ── Reset (execute UMA VEZ antes de começar do zero) ─────────────────
# Apaga pickle e checkpoints anteriores deste experimento.
# Comente o shutil se quiser preservar checkpoints já treinados.

import shutil

if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)
    print(f'Removido: {CHECKPOINT_FILE}')

if os.path.exists('./checkpoints'):
    shutil.rmtree('./checkpoints')
    print('Removido: ./checkpoints/')

print('Pronto para iniciar o grid search.')

In [ ]:
# ── Treinamento ───────────────────────────────────────────────────────
# Treina HoTHP para cada (lr, seed) com train_max_len.

progress = load_progress()
if 'train_times' not in progress:
    progress['train_times'] = {}

total_train = len(LEARNING_RATES) * len(SEEDS)
print(f'Treinamentos: {len(progress["trained"])}/{total_train} já concluídos\n')

for lr in LEARNING_RATES:
    for seed in SEEDS:
        key = (lr_tag(lr), seed)

        if key in progress['trained']:
            model_dir = get_model_dir(lr, seed)
            if os.path.exists(model_dir):
                t = progress['train_times'].get(key)
                t_str = f'{t:.0f}s ({t/60:.1f}min)' if t else 'N/A'
                print(f'  [SKIP] lr={lr_tag(lr)} seed={seed}  (treino: {t_str})')
                continue
            else:
                print(f'  [RETRAIN] lr={lr_tag(lr)} seed={seed} — checkpoint perdido, re-treinando...')
                progress['trained'].discard(key)
                progress['train_times'].pop(key, None)
                progress['results'] = [
                    r for r in progress['results']
                    if not (r['lr'] == lr_tag(lr) and r['seed'] == seed)
                ]
                save_progress(progress)

        train_len = DATASET_CFG['train_max_len']
        print(f'\n{"="*55}')
        print(f'  HoTHP | amazon | lr={lr_tag(lr)} | seed={seed} | max_len={train_len}')
        print(f'{"="*55}')

        cfg_dict, exp_id = make_yaml(lr, seed, max_len=train_len, stage='train')
        cfg = write_yaml_and_load(cfg_dict, experiment_id=exp_id)
        runner = Runner.build_from_config(cfg)

        t0 = time.time()
        runner.run()
        elapsed = time.time() - t0

        del runner
        free_gpu()

        progress['trained'].add(key)
        progress['train_times'][key] = elapsed
        save_progress(progress)
        print(f'  [OK] {elapsed:.0f}s ({elapsed/60:.1f}min)')

print(f'\nTreinamento concluído! {len(progress["trained"])}/{total_train}')
print('\nResumo de tempos:')
print(f'{"LR":<8} {"Seed":>6}  {"Tempo":>12}')
print('-' * 32)
for (lr_str, seed), t in sorted(progress['train_times'].items()):
    print(f'{lr_str:<8} {seed:>6}  {t:>7.0f}s ({t/60:.1f}min)')

In [ ]:
# ── Avaliação ─────────────────────────────────────────────────────────
# Avalia cada (lr, seed) nos cenários 1x e 5x.

progress = load_progress()

done_evals = set()
for r in progress['results']:
    done_evals.add((r['lr'], r['seed'], r['extrap']))

scenarios = [
    ('1x', DATASET_CFG['eval_1x']),
    ('5x', DATASET_CFG['eval_5x']),
]
total_evals = len(LEARNING_RATES) * len(SEEDS) * len(scenarios)
print(f'Avaliações: {len(done_evals)}/{total_evals} já concluídas\n')

for lr in LEARNING_RATES:
    for seed in SEEDS:
        model_dir = get_model_dir(lr, seed)
        if not os.path.exists(model_dir):
            print(f'  [WARN] Checkpoint não encontrado: lr={lr_tag(lr)} seed={seed}')
            continue

        for extrap_name, max_len in scenarios:
            eval_key = (lr_tag(lr), seed, extrap_name)
            if eval_key in done_evals:
                print(f'  [SKIP] lr={lr_tag(lr)} seed={seed} {extrap_name}')
                continue

            print(f'  Eval: lr={lr_tag(lr)} seed={seed} [{extrap_name}, max_len={max_len}]', end=' → ')

            cfg_dict, exp_id = make_yaml(
                lr, seed, max_len=max_len,
                stage='eval', pretrained_model_dir=model_dir)
            cfg = write_yaml_and_load(cfg_dict, experiment_id=exp_id)
            runner = Runner.build_from_config(cfg)

            test_loader = runner._data_loader.test_loader()
            metrics = runner._evaluate_model(test_loader)
            del runner
            free_gpu()

            result = {
                'lr':     lr_tag(lr),
                'lr_val': lr,
                'seed':   seed,
                'extrap': extrap_name,
                'max_len': max_len,
                'nll':    -metrics.get('loglike', float('nan')),
                'acc':    metrics.get('acc', float('nan')),
                'rmse':   metrics.get('rmse', float('nan')),
            }
            progress['results'].append(result)
            save_progress(progress)
            print(f'NLL={result["nll"]:.4f}  ACC={result["acc"]:.4f}  RMSE={result["rmse"]:.4f}')

print(f'\nAvaliação concluída! {len(progress["results"])} resultados salvos.')

In [ ]:
# ── Resultados ────────────────────────────────────────────────────────

progress = load_progress()
df = pd.DataFrame(progress['results'])
print(f'Total de resultados: {len(df)}')

# Agrega por (lr, extrap)
summary = df.groupby(['lr', 'lr_val', 'extrap']).agg(
    nll_mean=('nll', 'mean'), nll_std=('nll', 'std'),
    acc_mean=('acc', 'mean'), acc_std=('acc', 'std'),
    rmse_mean=('rmse', 'mean'), rmse_std=('rmse', 'std'),
).reset_index().sort_values('lr_val', ascending=False)

print('\n══════════════════════════════════════════════════════════════════')
print(f'GRID SEARCH — HoTHP | Amazon | {len(SEEDS)} seeds')
print('══════════════════════════════════════════════════════════════════')

for extrap_name in ['1x', '5x']:
    sub = summary[summary['extrap'] == extrap_name].copy()
    if len(sub) == 0:
        continue
    best_nll = sub['nll_mean'].min()
    print(f'\n── {extrap_name} (max_len={DATASET_CFG["eval_" + extrap_name]}) ──')
    print(f'{"LR":<8} {"NLL":>20} {"ACC":>18} {"RMSE":>18}')
    print('-' * 68)
    for _, row in sub.iterrows():
        marker = ' *' if abs(row['nll_mean'] - best_nll) < 1e-6 else '  '
        print(f'{row["lr"]:<8} '
              f'{row["nll_mean"]:>7.4f}+/-{row["nll_std"]:.4f}{marker} '
              f'{row["acc_mean"]:>7.4f}+/-{row["acc_std"]:.4f}  '
              f'{row["rmse_mean"]:>7.4f}+/-{row["rmse_std"]:.4f}')

print('\n* = melhor NLL no cenário')

In [ ]:
# ── Gráfico 1: NLL por LR (1x e 5x lado a lado) ──────────────────────

lr_labels = [lr_tag(lr) for lr in LEARNING_RATES]
x = np.arange(len(LEARNING_RATES))
bar_width = 0.35

# Paleta: 1x = mais escuro, 5x = mais claro
COLOR_1X = '#2166ac'
COLOR_5X = '#74add1'

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax_idx, (metric, ylabel, better) in enumerate([
    ('nll',  'NLL (menor = melhor)',      'min'),
    ('acc',  'Accuracy (maior = melhor)', 'max'),
    ('rmse', 'RMSE (menor = melhor)',     'min'),
]):
    ax = axes[ax_idx]

    for ei, (extrap_name, color) in enumerate([('1x', COLOR_1X), ('5x', COLOR_5X)]):
        sub = summary[summary['extrap'] == extrap_name].sort_values('lr_val', ascending=False)
        means = sub[f'{metric}_mean'].values
        stds  = sub[f'{metric}_std'].values
        offset = (ei - 0.5) * bar_width
        ax.bar(x + offset, means, bar_width,
               yerr=stds, capsize=4,
               label=extrap_name, color=color, alpha=0.88)

    ax.set_xticks(x)
    ax.set_xticklabels(lr_labels, fontsize=10)
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel(ylabel)
    ax.set_title(metric.upper(), fontweight='bold')
    ax.legend(title='Cenário')
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle(
    f'HoTHP — Grid Search LR | Amazon | {len(SEEDS)} seeds\n'
    f'train_len={DATASET_CFG["train_max_len"]}  '
    f'eval_1x={DATASET_CFG["eval_1x"]}  '
    f'eval_5x={DATASET_CFG["eval_5x"]}',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('hothp_lr_search_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: hothp_lr_search_metrics.png')

In [ ]:
# ── Gráfico 2: ΔNLL (5x − 1x) por LR ────────────────────────────────
# Quanto o modelo melhora ao ver sequências mais longas?
# Barra mais negativa = melhor aproveitamento de contexto longo.

delta_rows = []
for lr in LEARNING_RATES:
    for seed in SEEDS:
        sub = df[(df['lr'] == lr_tag(lr)) & (df['seed'] == seed)]
        nll_1x = sub[sub['extrap'] == '1x']['nll'].values
        nll_5x = sub[sub['extrap'] == '5x']['nll'].values
        if len(nll_1x) > 0 and len(nll_5x) > 0:
            delta_rows.append({
                'lr': lr_tag(lr), 'lr_val': lr, 'seed': seed,
                'delta_nll': nll_5x[0] - nll_1x[0]
            })

df_delta = pd.DataFrame(delta_rows)

fig, ax = plt.subplots(figsize=(8, 4))

delta_agg = df_delta.groupby(['lr', 'lr_val']).agg(
    mean=('delta_nll', 'mean'), std=('delta_nll', 'std')
).reset_index().sort_values('lr_val', ascending=False)

colors = ['#d73027' if m > 0 else '#4575b4' for m in delta_agg['mean']]
ax.bar(x, delta_agg['mean'], bar_width * 1.5,
       yerr=delta_agg['std'], capsize=5,
       color=colors, alpha=0.85)
ax.axhline(0, color='gray', ls='--', lw=1.2)

# Anota o valor em cima de cada barra
for i, (_, row) in enumerate(delta_agg.iterrows()):
    ax.text(i, row['mean'] - 0.004, f'{row["mean"]:.4f}',
            ha='center', va='top', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(delta_agg['lr'].values, fontsize=10)
ax.set_xlabel('Learning Rate')
ax.set_ylabel('ΔNLL = NLL(5x) − NLL(1x)')
ax.set_title(
    'Aproveitamento de Contexto Longo por LR\n'
    '(negativo = melhora ao extrapolar, azul = bom)',
    fontweight='bold'
)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('hothp_lr_search_delta_nll.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: hothp_lr_search_delta_nll.png')

In [ ]:
# ── Gráfico 3: NLL por seed (estabilidade) ────────────────────────────
# Cada ponto = 1 seed. Mostra variabilidade real entre seeds por LR.

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
markers = ['o', 's', '^']
seed_colors = ['#1b7837', '#762a83', '#e08214']

for ax_idx, extrap_name in enumerate(['1x', '5x']):
    ax = axes[ax_idx]
    sub = df[df['extrap'] == extrap_name].sort_values('lr_val', ascending=False)

    for si, seed in enumerate(SEEDS):
        seed_sub = sub[sub['seed'] == seed].sort_values('lr_val', ascending=False)
        ax.plot(range(len(LEARNING_RATES)), seed_sub['nll'].values,
                marker=markers[si], color=seed_colors[si],
                label=f'seed {seed}', linewidth=1.5, markersize=7)

    # Média
    means = [summary[(summary['extrap'] == extrap_name) &
                     (summary['lr'] == lr_tag(lr))]['nll_mean'].values
             for lr in LEARNING_RATES]
    means = [m[0] if len(m) > 0 else float('nan') for m in means]
    ax.plot(range(len(LEARNING_RATES)), means,
            color='black', linewidth=2.5, linestyle='--',
            label='média', zorder=5)

    ax.set_xticks(range(len(LEARNING_RATES)))
    ax.set_xticklabels(lr_labels, fontsize=10)
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('NLL')
    eval_len = DATASET_CFG[f'eval_{extrap_name}']
    ax.set_title(f'Cenário {extrap_name} (max_len={eval_len})', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f'Estabilidade entre Seeds — HoTHP | Amazon\n'
    f'(linha tracejada = média das 3 seeds)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('hothp_lr_search_stability.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: hothp_lr_search_stability.png')

In [ ]:
# ── Resumo final + tempo de treino ───────────────────────────────────

try:
    with open('hardware_info_lr_search.json') as f:
        hw = json.load(f)
except FileNotFoundError:
    hw = {k: 'N/A' for k in ['gpu_name', 'gpu_vram_gb', 'cuda_version', 'ram_gb', 'timestamp']}

train_times = progress.get('train_times', {})
time_rows = [{'lr': lr_str, 'seed': seed, 'time_s': t}
             for (lr_str, seed), t in train_times.items()]
df_times = pd.DataFrame(time_rows)

print('══════════════════════════════════════════════════════════════════')
print('TEMPOS DE TREINAMENTO')
print('══════════════════════════════════════════════════════════════════')
print(f'GPU  : {hw["gpu_name"]} ({hw["gpu_vram_gb"]} GB VRAM, CUDA {hw["cuda_version"]})')
print(f'RAM  : {hw["ram_gb"]} GB')
print(f'Data : {hw["timestamp"]}')
print()

if len(df_times) > 0:
    time_agg = df_times.groupby('lr').agg(
        mean_s=('time_s', 'mean'), std_s=('time_s', 'std'), total_s=('time_s', 'sum')
    ).reset_index()

    print(f'{"LR":<8} {"Média/seed":>16} {"Std":>8}  {"Total (3 seeds)":>18}')
    print('-' * 58)
    for _, row in time_agg.iterrows():
        print(f'{row["lr"]:<8} {row["mean_s"]:>8.0f}s ({row["mean_s"]/60:>4.1f}min)  '
              f'{row["std_s"]:>4.0f}s  '
              f'{row["total_s"]:>8.0f}s ({row["total_s"]/60:>5.1f}min)')

    total_all = df_times['time_s'].sum()
    print(f'\nTOTAL: {total_all:.0f}s ({total_all/3600:.2f}h)')

# Melhor LR
print('\n══════════════════════════════════════════════════════════════════')
print('MELHOR LR POR CENÁRIO')
print('══════════════════════════════════════════════════════════════════')
for extrap_name in ['1x', '5x']:
    sub = summary[summary['extrap'] == extrap_name]
    if len(sub) == 0:
        continue
    best = sub.loc[sub['nll_mean'].idxmin()]
    print(f'  {extrap_name}: LR={best["lr"]}  NLL={best["nll_mean"]:.4f}±{best["nll_std"]:.4f}  '
          f'ACC={best["acc_mean"]:.4f}  RMSE={best["rmse_mean"]:.4f}')